# LoRA vs DoRA Benchmark Comparison

In [ ]:
import json
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np

# Add or remove models here — everything downstream adapts automatically.
RESULTS_ROOT = "/teamspace/lightning_storage/evals"

MODELS = {
    "Base":        f"{RESULTS_ROOT}/qwen2.5-1.5b-base-v2/results.json",
    "QLoRA 4-bit": f"{RESULTS_ROOT}/qwen2.5-1.5b-qlora-4bit-v2/results.json",
    "QDoRA 4-bit": f"{RESULTS_ROOT}/qwen2.5-1.5b-qdora-4bit-v2/results.json",
    "QLoRA 8-bit": f"{RESULTS_ROOT}/qwen2.5-1.5b-qlora-8bit-v2/results.json",
    "QDoRA 8-bit": f"{RESULTS_ROOT}/qwen2.5-1.5b-qdora-8bit-v2/results.json",
}

COLORS = ["#999999", "#DD8452", "#4C72B0", "#E07B54", "#5A9BD5"]

In [ ]:
rows = []
for model_name, path in MODELS.items():
    with open(path) as f:
        raw = json.load(f)
    for bench, metrics in raw["results"].items():
        rows.append({
            "model": model_name,
            "benchmark": bench,
            "acc": metrics.get("acc,none", metrics.get("acc")),
            "acc_norm": metrics.get("acc_norm,none", metrics.get("acc_norm")),
            "acc_stderr": metrics.get("acc_stderr,none", metrics.get("acc_stderr")),
            "acc_norm_stderr": metrics.get("acc_norm_stderr,none", metrics.get("acc_norm_stderr")),
        })

df = pd.DataFrame(rows)
print(f"Loaded {df['model'].nunique()} models, {df['benchmark'].nunique()} benchmarks")
df.head()

## Accuracy Comparison Table

In [ ]:
pivot = df.pivot(index="benchmark", columns="model", values="acc")[list(MODELS.keys())]
avg = pivot.mean().rename("Average")
table = pd.concat([pivot, avg.to_frame().T])
table.style.format("{:.2%}").background_gradient(cmap="YlGn", axis=1)

## Accuracy by Benchmark

In [ ]:
model_names = list(MODELS.keys())
benchmarks = df["benchmark"].unique()
n_models = len(model_names)
n_bench = len(benchmarks)
x = np.arange(n_bench)
width = 0.8 / n_models

fig, ax = plt.subplots(figsize=(14, 6))
for i, model in enumerate(model_names):
    sub = df[df["model"] == model].set_index("benchmark").loc[benchmarks]
    bars = ax.bar(x + i * width, sub["acc"], width,
                  yerr=sub["acc_stderr"], capsize=3,
                  label=model, color=COLORS[i % len(COLORS)])

ax.set_ylabel("Accuracy")
ax.set_title("Accuracy by Benchmark")
ax.set_xticks(x + width * (n_models - 1) / 2)
ax.set_xticklabels([b.replace("_", " ").title() for b in benchmarks], rotation=30, ha="right")
ax.legend()
ax.grid(axis="y", alpha=0.3)
plt.tight_layout()
plt.show()

## Normalized Accuracy by Benchmark

In [ ]:
df_norm = df.dropna(subset=["acc_norm"])
norm_benchmarks = df_norm["benchmark"].unique()
n_nb = len(norm_benchmarks)
x = np.arange(n_nb)
width = 0.8 / n_models

fig, ax = plt.subplots(figsize=(14, 6))
for i, model in enumerate(model_names):
    sub = df_norm[df_norm["model"] == model].set_index("benchmark").loc[norm_benchmarks]
    ax.bar(x + i * width, sub["acc_norm"], width,
           yerr=sub["acc_norm_stderr"], capsize=3,
           label=model, color=COLORS[i % len(COLORS)])

ax.set_ylabel("Normalized Accuracy")
ax.set_title("Normalized Accuracy by Benchmark")
ax.set_xticks(x + width * (n_models - 1) / 2)
ax.set_xticklabels([b.replace("_", " ").title() for b in norm_benchmarks], rotation=30, ha="right")
ax.legend()
ax.grid(axis="y", alpha=0.3)
plt.tight_layout()
plt.show()

## Delta vs Base (Accuracy Improvement Heatmap)

In [ ]:
adapter_models = [m for m in model_names if m != "Base"]
base_acc = pivot["Base"]

delta = pivot[adapter_models].sub(base_acc, axis=0)

fig, ax = plt.subplots(figsize=(10, 6))
im = ax.imshow(delta.values, cmap="RdYlGn", aspect="auto", vmin=-0.05, vmax=0.05)

ax.set_xticks(range(len(adapter_models)))
ax.set_xticklabels(adapter_models, rotation=30, ha="right")
ax.set_yticks(range(len(delta.index)))
ax.set_yticklabels([b.replace("_", " ").title() for b in delta.index])

for i in range(len(delta.index)):
    for j in range(len(adapter_models)):
        ax.text(j, i, f"{delta.values[i, j]:+.1%}", ha="center", va="center", fontsize=10)

ax.set_title("Accuracy Change vs Base Model")
fig.colorbar(im, ax=ax, label="Δ Accuracy")
plt.tight_layout()
plt.show()

## Summary

In [ ]:
print("=" * 55)
print("AVERAGE ACCURACY ACROSS ALL BENCHMARKS")
print("=" * 55)
for model in model_names:
    avg = df[df["model"] == model]["acc"].mean()
    print(f"  {model:15s}  {avg:.2%}")

print()
print("Per-benchmark winner:")
for bench in benchmarks:
    sub = df[df["benchmark"] == bench].set_index("model")["acc"]
    winner = sub.idxmax()
    best = sub.max()
    print(f"  {bench:15s}  {best:.2%}  ({winner})")